# Alignment

Alignment는 AI 모델이 사람의 의도, 가치, 선호에 맞는 응답을 생성하도록 조정하는 과정이다.

단순히 다음 단어를 잘 예측하는 모델을 만드는 것을 넘어, 사람이 더 선호하는 답변, 안전한 답변, 지시를 잘 따르는 답변을 생성하도록 모델을 정렬하는 것이 목적이다.

대표적인 Alignment 기법에는 RLHF, DPO, RRHF, RLAIF 등이 있다.

---

## 1. RLHF (Reinforcement Learning from Human Feedback)

RLHF는 사람의 피드백을 이용해 모델을 강화학습으로 정렬하는 대표적인 방식이다.

사람이 여러 응답을 비교하여 더 좋은 응답을 선택하고, 이 선호 데이터를 바탕으로 보상 모델(Reward Model)을 학습한 뒤, PPO 같은 강화학습 알고리즘을 사용해 모델을 조정한다.

### 작동 흐름

1. 선호도 데이터 수집
   - 하나의 질문에 대해 여러 응답을 생성한다.
   - 사람이 응답을 비교하여 더 선호하는 답변을 선택한다.

2. 보상 모델 학습
   - 사람의 선택 기준을 학습하는 보상 모델을 만든다.
   - 보상 모델은 응답이 얼마나 좋은지 점수화한다.

3. 강화학습 적용
   - 보상 모델의 점수를 최대화하도록 PPO 알고리즘으로 모델을 조정한다.

### 장점

- 사람의 선호를 직접 반영할 수 있다.
- 모델이 더 자연스럽고 유용한 답변을 생성하도록 조정할 수 있다.
- 초기 ChatGPT 계열 모델 정렬 방식으로 널리 알려졌다.

### 단점

- 사람의 선호도 데이터를 수집하는 데 비용과 시간이 많이 든다.
- 보상 모델을 따로 학습해야 한다.
- PPO 기반 학습 과정이 복잡하고 불안정할 수 있다.
- 보상 모델의 품질에 따라 최종 성능이 크게 달라진다.

---

## 2. DPO (Direct Preference Optimization, 직접 선호 최적화)

DPO는 복잡한 강화학습 과정을 생략하고, 사람의 선호 데이터를 이용해 모델을 직접 학습하는 방식이다.

RLHF처럼 별도의 보상 모델을 만들고 PPO를 적용하는 대신, 선호된 응답과 덜 선호된 응답의 차이를 손실 함수에 직접 반영한다.

### 핵심 아이디어

별도의 보상 모델이나 강화학습 없이, 선호도 데이터를 직접 학습 목표로 사용한다.

예시는 다음과 같다.

- 질문: x
- 선호 응답: y_winner
- 비선호 응답: y_loser

모델은 y_winner를 선택할 확률은 높이고, y_loser를 선택할 확률은 낮추도록 학습된다.

### 작동 방식

1. 하나의 질문에 대해 선호 응답(Winner)과 비선호 응답(Loser)을 준비한다.
2. 모델이 Winner를 더 높은 확률로 생성하도록 학습한다.
3. 동시에 Loser를 선택할 확률은 낮추도록 유도한다.
4. 이 과정을 DPO 손실 함수로 직접 최적화한다.

### 장점

- RLHF보다 구조가 단순하다.
- 별도의 보상 모델이 필요 없다.
- PPO 강화학습 과정을 생략할 수 있다.
- 학습이 상대적으로 안정적이다.
- 구현이 비교적 쉽고, 현재 LLM 정렬 실습에서 자주 사용된다.

### 단점

- 새로운 답변을 탐색(Exploration)하는 능력은 제한적이다.
- 주어진 선호 데이터의 품질에 크게 의존한다.
- 데이터가 편향되어 있으면 모델도 그 편향을 학습할 수 있다.

---

## 3. RRHF (Reward-Rank Hindsight Fine-Tuning)

RRHF는 생성된 여러 응답에 대해 보상 점수를 매기고, 그 순위에 맞게 모델을 정렬하는 방식이다.

RLHF처럼 복잡한 PPO 강화학습을 사용하지 않고, 응답의 순위 정보를 활용하여 모델을 미세 조정한다.

### 핵심 아이디어

생성된 답변들에 대해 보상 점수를 매긴 뒤, 높은 점수를 받은 응답을 더 잘 생성하도록 모델을 학습한다.

즉, 모델은 단순히 하나의 정답만 배우는 것이 아니라, 여러 후보 응답 사이의 상대적인 품질 차이를 학습한다.

### 작동 방식

1. 하나의 질문에 대해 다양한 응답을 생성한다.
2. 보상 모델 또는 평가 기준을 사용해 각 응답에 점수를 매긴다.
3. 점수에 따라 응답의 순위를 정한다.
4. 높은 점수를 받은 응답을 더 잘 생성하도록 모델을 업데이트한다.

### 장점

- PPO를 사용하지 않아 RLHF보다 단순하다.
- RLHF 대비 메모리 사용량이 적고 효율적이다.
- 여러 응답의 순위 정보를 학습에 활용할 수 있다.

### 단점

- 여전히 보상 모델 또는 평가 기준의 품질에 크게 의존한다.
- 학습 과정에서 여러 응답을 동시에 처리해야 하므로 순간적인 GPU 메모리 사용량이 커질 수 있다.
- DPO에 비해 현재 실무·교육 현장에서는 상대적으로 덜 일반적이다.

---

## 4. RLAIF (Reinforcement Learning from AI Feedback)

RLAIF는 사람이 직접 피드백을 제공하는 대신, AI가 피드백을 제공하여 모델을 정렬하는 방식이다.

즉, 사람의 평가를 AI 평가로 대체하거나 보완하여 강화학습 또는 선호 학습에 활용한다.

### 핵심 아이디어

사람이 모든 선호 데이터를 직접 만드는 것은 비용과 시간이 많이 든다.

따라서 강력한 LLM을 평가자 역할로 사용해, 응답 간 선호도를 판단하게 한다.

### 작동 방식

1. 하나의 질문에 대해 여러 응답을 생성한다.
2. 평가자 역할을 하는 AI에게 어떤 응답이 더 나은지 판단하도록 한다.
3. AI가 생성한 선호도 데이터를 바탕으로 모델을 학습한다.
4. 필요에 따라 RLHF, DPO, RRHF 등의 방식과 결합할 수 있다.

### 장점

- 사람의 데이터 수집 비용과 시간을 줄일 수 있다.
- 대규모 선호도 데이터 생성이 가능하다.
- 반복적인 평가 작업을 자동화하기 쉽다.
- 인간 피드백이 부족한 상황에서 유용하다.

### 단점

- 평가자 AI의 성능에 따라 결과 품질이 달라진다.
- 평가 프롬프트 설계가 중요하다.
- AI의 편향이 학습 데이터에 반영될 위험이 있다.
- 잘못된 평가 기준이 누적되면 모델 품질이 오히려 떨어질 수 있다.

---

## 5. 요약 및 결론

각 Alignment 기법은 상황과 목적에 따라 선택해서 사용해야 한다.

| 구분 | 추천 상황 | 특징 |
|---|---|---|
| RLHF | 고품질 인간 선호도 데이터가 있고, 충분한 학습 자원이 있을 때 | 사람의 선호를 직접 반영하지만 구현과 학습 과정이 복잡하다. |
| DPO | 선호 응답과 비선호 응답 데이터가 있고, 안정적이고 단순한 정렬 학습을 원할 때 | 보상 모델과 PPO 없이 선호 데이터를 직접 학습할 수 있다. |
| RRHF | 여러 후보 응답의 순위 정보를 활용하고 싶을 때 | 응답의 상대적 품질 차이를 학습할 수 있으며 PPO보다 단순하다. |
| RLAIF | 인간 평가 비용을 줄이고 대규모 선호 데이터를 만들고 싶을 때 | AI가 평가자 역할을 하며 확장성이 좋지만 평가자 AI의 품질에 의존한다. |

### 실무적 선택 기준

| 상황 | 우선 고려할 방법 |
|---|---|
| 수업 실습 또는 빠른 구현 | DPO |
| 대규모 서비스 수준의 정렬 | RLHF 또는 RLAIF + DPO |
| 사람이 직접 평가하기 어려운 대량 데이터 | RLAIF |
| 여러 응답의 순위 정보를 활용 | RRHF |
| 복잡한 강화학습을 피하고 싶음 | DPO 또는 RRHF |

### 핵심 정리

- RLHF는 사람의 피드백을 가장 직접적으로 반영하지만 비용과 구현 난이도가 높다.
- DPO는 RLHF의 복잡한 강화학습 과정을 줄이고, 선호 데이터를 직접 학습하는 실용적인 방식이다.
- RRHF는 여러 응답의 순위 정보를 활용해 모델을 조정한다.
- RLAIF는 사람 대신 AI가 평가자로 참여하여 데이터 생성 비용을 줄인다.

결국 모든 Alignment 기법의 목표는 같다.

> AI가 단순히 그럴듯한 문장을 생성하는 것을 넘어, 사람의 의도와 선호에 더 잘 맞는 응답을 생성하도록 만드는 것이다.
